# Stage 02a-legacy — codebook conformance for Batches 01 and 05

Brings the two batches labelled under older wizard/codebook revisions up to the current
specification, **without re-labelling them**, and writes the result to the corrected working
copy that `02a_preprocessing` consumes.

```
03_HUMAN_wizard_exports/      frozen: chmod 444 + PRISTINE_MANIFEST.sha256   ← never written
   │  freeze_exports.py --copy
   ▼
03c_CORRECTED_wizard_exports/ ← THIS NOTEBOOK READS AND WRITES HERE
   │  02a_preprocessing.ipynb
   ▼
02a_CLEANED_label_tables/     Review_postprocess_<batch>_<ts>.xlsx  → 02b
```

## How the conformance is done

Not by a hand-written rename map. Each patent is pushed back through **the wizard's own
code** (`ingestPatentRows → buildExport → recordToRows`, run headless in Node) and re-exported.
Whatever the wizard migrates, it migrates with the same logic a reviewer would trigger by
opening the patent — which is by definition the schema-current answer. The harmonisation log is
then generated from the *observed* diff rather than from a specification of what should have
happened.

**The wizard is trusted for `G1`/`M1`/`M2`/`M3` only.** `T1`/`T2`/`META` are carried through
verbatim. Measured 2026-08-19 over an 8-patent sample, the T2 figure round-trip is **not**
lossless: 4 of 8 dropped per-figure labels outright, and one had `T2_APPEARANCE_DEFAULTS`
re-applied over the annotator's own picks (`Top→Front-Isometric`, `Render→Line Drawing`,
`Grayscale→B/W`). Nothing in this conformance touches a figure label, so those rows are not
regenerated at all. See `03b_CONFORMED_legacy/STEP1_roundtrip_report.md`.

## What this notebook does NOT do

- It never writes to `03_HUMAN_wizard_exports/`. Section 1 installs a guard that raises if
  anything tries.
- It does not decide anything that needs a figure. Those become worklist rows in
  `03b_CONFORMED_legacy/proposals/` for the propose → confirm → apply pass.
- It does not migrate anything to or from `BodyPitch` while the codebook's 3-option vs the
  wizard's 4-option divergence is open (Section 6).

## Section 1 — Imports, config, and the write guard

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from src.config_loader import load_config

cfg = load_config()

BATCH = "Batch_05"          # <- Batch_01 or Batch_05. This notebook is only for those two.

FROZEN_DIR    = Path(cfg["paths"]["html_review_exports"])
CORRECTED_DIR = Path(cfg["paths"]["corrected_wizard_exports"])
CONFORM_DIR   = Path(cfg["paths"]["base"]) / "data" / "03b_CONFORMED_legacy"
PROPOSALS_DIR = CONFORM_DIR / "proposals"
HARNESS       = repo_root / "scripts" / "legacy_roundtrip_harness" / "conform_one.js"

for d in (PROPOSALS_DIR,):
    d.mkdir(parents=True, exist_ok=True)

assert BATCH in ("Batch_01", "Batch_05"), (
    f"{BATCH} is not a legacy batch. Batch_02/03/04 were labelled on v15_3 and go straight "
    "through 02a_preprocessing."
)

# ── The write guard ───────────────────────────────────────────────────────────
# The frozen exports are the only record of what the annotator actually clicked, and
# intra-annotator kappa is measured on them. They are chmod 444, but a notebook run as the
# owner can still chmod them back, so belt and braces: wrap the two functions that could
# plausibly write there and refuse by path. This fires on the DIRECTORY, not the file mode,
# so it also catches a new file being created alongside the originals.
_real_to_excel = pd.DataFrame.to_excel
_real_open = open

def _assert_not_frozen(target):
    try:
        p = Path(target).resolve()
    except (TypeError, ValueError):
        return
    if FROZEN_DIR.resolve() in p.parents:
        raise PermissionError(
            f"REFUSED: {p.name} is inside {FROZEN_DIR.name}/, which is immutable.\n"
            f"Every correction belongs in {CORRECTED_DIR.name}/. "
            f"Rebuild a working copy with:\n"
            f"    python scripts/freeze_exports.py --copy {BATCH} --force"
        )

def _guarded_to_excel(self, excel_writer, *a, **kw):
    if isinstance(excel_writer, (str, Path)):
        _assert_not_frozen(excel_writer)
    return _real_to_excel(self, excel_writer, *a, **kw)

pd.DataFrame.to_excel = _guarded_to_excel

print(f"batch            : {BATCH}")
print(f"frozen (read-only): {FROZEN_DIR}")
print(f"corrected (r/w)   : {CORRECTED_DIR}")
print(f"harness           : {HARNESS}  {'OK' if HARNESS.exists() else 'MISSING'}")
print("write guard       : armed")

## Section 2 — Integrity gate

Refuses to run if any frozen export has changed since it was hashed. A mismatch means an
export was edited in place, and every number downstream would be measured against the wrong
baseline. Fix that before anything else.

In [ ]:
_verify = subprocess.run(
    [sys.executable, str(repo_root / "scripts" / "freeze_exports.py"), "--verify"],
    capture_output=True, text=True, cwd=repo_root,
)
print(_verify.stdout.strip() or _verify.stderr.strip())
if _verify.returncode != 0:
    raise SystemExit(
        "Frozen-export verification FAILED — stop here and restore from backup.\n"
        "A mismatch means one of the human exports was edited in place."
    )

# Node is the conformance engine; fail loudly and early rather than 200 patents in.
_node = subprocess.run(["node", "--version"], capture_output=True, text=True)
assert _node.returncode == 0, "node not found on PATH — the conformance engine needs it."
print(f"node             : {_node.stdout.strip()}")

## Section 3 — Load the working copy

Reads from `03c_CORRECTED_wizard_exports/`, **not** from the frozen original. That folder may
already carry corrections this notebook must not revert — Batch_05's copy, for instance,
received the cross-batch duplicate link for `US2008283673A1 → US2004155143A1` on 2026-08-19
(3 rows, see `CORRECTIONS_LOG.csv`), which the wizard cannot record during review because the
original lives in another batch.

If no working copy exists yet, this cell stops and tells you the command to make one, rather
than silently falling back to the frozen file.

In [ ]:
src_xlsx = CORRECTED_DIR / f"reviewed_patents_{BATCH}.xlsx"
if not src_xlsx.exists():
    raise SystemExit(
        f"No working copy at {src_xlsx}\n"
        f"Make one from the frozen original first:\n"
        f"    python scripts/freeze_exports.py --copy {BATCH}"
    )

df_raw = pd.read_excel(src_xlsx, sheet_name="Review")
COLS = ["Patent_ID", "Section", "Sub_Dimension", "Field", "Value", "Source", "Image_Path"]
df_raw = df_raw[COLS]

def base_id(pid: str) -> str:
    """Strip the _archN suffix. Multi-architecture patents export one row-set per
    architecture but are ONE patent for load/ingest purposes."""
    return re.sub(r"_arch\d+$", "", str(pid))

def strip_label(v):
    """Export Values are "ID — Label" composites (withLabel). Mirrors the wizard's
    stripLabel(); always compare on the id, never the display half — the display strings
    differ between batches for identical ids."""
    s = str(v)
    return s.split(" — ")[0].strip() if " — " in s else s.strip()

df_raw["base"] = df_raw.Patent_ID.map(base_id)
patents = sorted(df_raw.base.unique())

print(f"{src_xlsx.name}: {len(df_raw):,} rows, {len(patents)} patents, "
      f"{df_raw.Patent_ID.nunique()} arch-records")
print(f"sections: {df_raw.Section.value_counts().to_dict()}")

## Section 4 — Round-trip conformance

One patent per process. Two non-obvious constraints, both learned the hard way and both
implemented inside `conform_one.js`:

1. **`S = fresh()` before every `ingestPatentRows()`**, mirroring `loadBatchPatent()`. Without
   it, stale module state suppresses the entire per-figure T2 export and the loss is silent.
2. **Fork per patent.** `S.archProfiles` persists between `ingestPatentRows()` calls, so a
   single-architecture patent ingested after a multi-architecture one is re-exported with
   `_arch1` suffixes it should not have (`recordToRows` keys off `archs.length`).

A patent whose Patent_IDs do not come back identical is **not** conformed — it is left at its
original rows and reported. That check is the tripwire for constraint 2 regressing.

In [ ]:
def conform_patent(rows: list[dict]) -> dict:
    """Push one patent's rows through the wizard and return {rows, changed, dropped, added}.

    Morphology (G1/M1/M2/M3) comes back from the wizard; T1/T2/META are carried verbatim —
    see the notebook header for why the T2 round-trip is not trusted."""
    tmp = Path(os.environ.get("TMPDIR", "/tmp")) / f"_conform_{os.getpid()}.json"
    tmp.write_text(json.dumps(rows, default=str))
    try:
        proc = subprocess.run(["node", str(HARNESS), str(tmp)],
                              capture_output=True, text=True, cwd=HARNESS.parent)
        if proc.returncode != 0 or not proc.stdout.strip():
            return {"error": (proc.stderr or "no stdout").strip()[:400]}
        return json.loads(proc.stdout)
    finally:
        tmp.unlink(missing_ok=True)

# Patents the annotator DISAPPROVED are carried through verbatim and never conformed.
# recordToRows() deliberately truncates a disapproved record to the verdict + reason
# ("everything else below would just be unreviewed ML/placeholder noise"), which is right
# for the wizard but destructive here: 2 B05 patents carry a human topType = HB behind an
# isApproved = False, and conforming them would silently drop 36 morphology rows AND log
# them as "field retired", which they are not. Their labels never reach 02b either way,
# so there is nothing to gain and provenance to lose.
_appr = df_raw[df_raw.Field == "isApproved"].copy()
_appr["base"] = _appr.Patent_ID.map(base_id)
disapproved = set(_appr.loc[_appr.Value.astype(str).str.lower() == "false", "base"])
print(f"disapproved (carried verbatim, not conformed): {len(disapproved)}")

results, conformed_rows, failures, skipped = {}, [], [], []

for i, pid in enumerate(patents, 1):
    g = df_raw[df_raw.base == pid]
    rows = g[COLS].where(pd.notna(g[COLS]), None).to_dict("records")

    if pid in disapproved:
        skipped.append(pid)
        conformed_rows.extend(rows)
        continue

    res = conform_patent(rows)

    if res.get("error") or set(res.get("inPids", [])) != set(res.get("outPids", [])):
        # Not conformed. Keep the original rows untouched — a partially-migrated patent is
        # worse than an unmigrated one, because nothing downstream would flag it.
        failures.append({"patent": pid,
                         "reason": res.get("error") or
                                   f"arch mismatch in={res.get('inPids')} out={res.get('outPids')}"})
        conformed_rows.extend(rows)
        continue

    results[pid] = res
    conformed_rows.extend(res["rows"])

    if i % 25 == 0 or i == len(patents):
        print(f"  {i:4d}/{len(patents)} conformed", flush=True)

df_conf = pd.DataFrame(conformed_rows)[COLS]
print(f"\nconformed : {len(results)}/{len(patents)} patents")
print(f"skipped   : {len(skipped)} (disapproved — carried verbatim)")
print(f"failed    : {len(failures)}")
print(f"rows      : {len(df_raw):,} in -> {len(df_conf):,} out")
for f in failures[:10]:
    print(f"  !! {f['patent']}: {f['reason'][:150]}")

### 4b — what the wizard actually changed

In [ ]:
import collections

chg = collections.Counter()
drp = collections.Counter()
add = collections.Counter()
examples: dict[str, tuple] = {}

for pid, r in results.items():
    for c in r["changed"]:
        chg[c["f"]] += 1
        examples.setdefault(c["f"], (c["from"], c["to"]))
    for d in r["dropped"]:
        drp[d["f"]] += 1
    for a in r["added"]:
        add[a["f"]] += 1

print("CHANGED — value differs after the round-trip")
for f, n in chg.most_common():
    a, b = examples[f]
    print(f"  {f:26s} n={n:4d}   {a[:34]!r} -> {b[:34]!r}")
print("\nDROPPED — retired field, no longer exported")
for f, n in drp.most_common():
    print(f"  {f:26s} n={n:4d}")
print("\nADDED — field introduced after this batch was labelled (default written)")
for f, n in add.most_common(20):
    print(f"  {f:26s} n={n:4d}")

## Section 5 — The rules the wizard does not cover

Three deterministic rewrites the round-trip leaves alone, because they are patent-level `T1`/
`T2`/`META` values and this notebook deliberately carries that half through verbatim.

| rule | scope | what |
|---|---|---|
| **L-1** | B05, 5 patents | `t1EdgeTags` — drop the `Tailsitter` token. The `TB` architecture already encodes it. Two of the five carry `Tailsitter\|UAVSimilar`, so strip the token and keep the rest — do not clear the field |
| **L-2** | B05, 1 patent | `t1EdgeTags = OutOfScope` → the disapproval reason `Out of Domain`, which is where it belongs now |
| **L-3** | B01, 1 patent | `qualityFlag = draft` → `acSty = Draft`. Draft describes a rendering STYLE, not an image-quality defect, and moved lists in v15.3 |

`fusShape = PodBoom` (B05, 1) is **not** here: choosing the base shape underneath the pod-and-boom
needs the figure, so it is a worklist row in Section 8.

In [ ]:
hand_rules = []   # one dict per applied change, for the harmonisation log

def _log(patent_id, section, field, old, new, rule, evidence):
    hand_rules.append(dict(patent_id=patent_id, section=section, field=field,
                           old_value=old, new_value=new, rule=rule, evidence=evidence))

# ── L-1 — drop the Tailsitter edge tag, preserving any sibling tokens ────────
mask = (df_conf.Field == "t1EdgeTags") & df_conf.Value.astype(str).str.contains("Tailsitter", na=False)
for idx in df_conf.index[mask]:
    old = str(df_conf.at[idx, "Value"])
    kept = [t for t in old.split("|") if t.strip() != "Tailsitter"]
    new = "|".join(kept)
    df_conf.at[idx, "Value"] = new or None
    _log(df_conf.at[idx, "Patent_ID"], "META", "t1EdgeTags", old, new or "(cleared)",
         "L-1 Tailsitter tag retired", "TB architecture already encodes tailsitter; token dropped, siblings kept")

# ── L-2 — OutOfScope edge tag becomes a disapproval reason ───────────────────
mask = (df_conf.Field == "t1EdgeTags") & (df_conf.Value.astype(str).str.strip() == "OutOfScope")
for idx in df_conf.index[mask]:
    pid = df_conf.at[idx, "Patent_ID"]
    df_conf.at[idx, "Value"] = None
    _log(pid, "META", "t1EdgeTags", "OutOfScope", "(cleared)",
         "L-2 OutOfScope retired", "superseded by the Out of Domain disapproval reason")
    # Only set the reason if the patent does not already carry one — never overwrite a
    # reason the annotator actually chose.
    has = ((df_conf.Patent_ID == pid) & (df_conf.Field == "t1DisapproveReason")
           & df_conf.Value.notna()).any()
    if has:
        rows_i = df_conf.index[(df_conf.Patent_ID == pid) & (df_conf.Field == "t1DisapproveReason")]
        print(f"  L-2: {pid} already has a disapproval reason "
              f"({df_conf.at[rows_i[0], 'Value']!r}) — left as is")
    else:
        df_conf.loc[len(df_conf)] = {
            "Patent_ID": pid, "Section": "T1", "Sub_Dimension": "T1 — Disapproval Reason",
            "Field": "t1DisapproveReason", "Value": "Out of Domain — Out of Domain",
            "Source": "human", "Image_Path": None,
        }
        _log(pid, "T1", "t1DisapproveReason", "(absent)", "Out of Domain",
             "L-2 OutOfScope retired", "edge tag OutOfScope carried the same meaning")

# ── L-3 — Draft moves from image quality to rendering style ─────────────────
mask = (df_conf.Field == "qualityFlag") & (df_conf.Value.map(strip_label) == "draft")
for idx in df_conf.index[mask]:
    pid, sub = df_conf.at[idx, "Patent_ID"], df_conf.at[idx, "Sub_Dimension"]
    df_conf.at[idx, "Value"] = "clean — Clean"
    _log(pid, "T2", "qualityFlag", "draft — Draft", "clean — Clean",
         "L-3 Draft is a style not a defect", f"{sub}: quality reset to clean, style set to Draft")
    sty = df_conf.index[(df_conf.Patent_ID == pid) & (df_conf.Sub_Dimension == sub)
                        & (df_conf.Field == "acSty")]
    if len(sty):
        _log(pid, "T2", "acSty", str(df_conf.at[sty[0], "Value"]), "Draft",
             "L-3 Draft is a style not a defect", f"{sub}: moved from qualityFlag")
        df_conf.at[sty[0], "Value"] = "Draft"

print(f"hand rules applied: {len(hand_rules)}")
for h in hand_rules:
    print(f"  {h['rule'][:34]:34s} {h['patent_id']:24s} {h['field']:20s} "
          f"{str(h['old_value'])[:22]!r} -> {str(h['new_value'])[:22]!r}")

## Section 6 — Quarantine: `VarInc` on whole-body-pitch architectures

`VarInc` is retired and needs a re-pick, but on `MR`/`PTC`/`RC` the candidate answers include
`BodyPitch` — and the codebook's Fuselage Kinematics has three options while the wizard has four
(it adds Whole-Body Pitch). That divergence is being resolved separately, and until it is,
**nothing migrates to or from `BodyPitch`.**

So these records are listed and left alone. Verified 2026-08-19: all of them are `MR`, which left
`BODYPITCH_ARCHS` on 2026-08-18, so no lock fires and the round-trip passes `VarInc` through
untouched. The quarantine holds by itself — this cell only proves it did.

In [ ]:
top_by_pid  = df_conf[df_conf.Field == "topType"].set_index("Patent_ID").Value.map(strip_label)
fus_by_pid  = df_conf[df_conf.Field == "fusKin"].set_index("Patent_ID").Value.map(strip_label)
BODYPITCH_ARCHS = ["PTC", "RC"]     # mirrors the wizard; MR left this list 2026-08-18

quarantined = [
    dict(patent_id=pid, topType=top_by_pid.get(pid), fusKin=v)
    for pid, v in fus_by_pid.items()
    if v == "VarInc" and top_by_pid.get(pid) in ("MR", *BODYPITCH_ARCHS)
]

print(f"quarantined (left at VarInc, reported as a coverage limitation): {len(quarantined)}")
for q in quarantined:
    print(f"  {q['patent_id']:26s} topType={q['topType']:4s} fusKin={q['fusKin']}")

violations = [q for q in quarantined if q["fusKin"] == "BodyPitch"]
assert not violations, f"BodyPitch was written to a quarantined record: {violations}"
print("\nno record migrated to or from BodyPitch — constraint holds")

## Section 7 — Harmonisation log

Appendix A of the thesis. One row per field per patent: what it was, what it became, which rule
did it, and whether that rule was deterministic. Rows come from two places — the observed diff of
the wizard round-trip (Section 4) and the hand rules (Section 5) — and are labelled so a reader
can tell which is which.

In [ ]:
log_rows = []

for pid, r in results.items():
    for c in r["changed"]:
        log_rows.append(dict(batch=BATCH, patent_id=c["pid"], field=c["f"],
                             old_value=c["from"], new_value=c["to"],
                             rule="wizard round-trip (v15_3 ingest/export)",
                             deterministic=True, source="observed"))
    for d in r["dropped"]:
        log_rows.append(dict(batch=BATCH, patent_id=d["pid"], field=d["f"],
                             old_value=d["v"], new_value="(field retired)",
                             rule="wizard round-trip (v15_3 ingest/export)",
                             deterministic=True, source="observed"))
    for a in r["added"]:
        log_rows.append(dict(batch=BATCH, patent_id=a["pid"], field=a["f"],
                             old_value="(field did not exist)", new_value=a["v"],
                             rule="wizard round-trip (v15_3 ingest/export)",
                             deterministic=True, source="observed"))

for h in hand_rules:
    log_rows.append(dict(batch=BATCH, patent_id=h["patent_id"], field=h["field"],
                         old_value=h["old_value"], new_value=h["new_value"],
                         rule=h["rule"], deterministic=True, source="rule"))

for q in quarantined:
    log_rows.append(dict(batch=BATCH, patent_id=q["patent_id"], field="fusKin",
                         old_value=q["fusKin"], new_value="(unchanged — quarantined)",
                         rule="BodyPitch divergence open; not migrated",
                         deterministic=False, source="quarantine"))

df_log = pd.DataFrame(log_rows)
log_path = CONFORM_DIR / f"harmonisation_log_{BATCH}.csv"
df_log.to_csv(log_path, index=False)

print(f"{log_path.name}: {len(df_log):,} rows")
if len(df_log):
    print(df_log.groupby(["source", "rule"]).size().to_string())

## Section 8 — Worklists

Everything that needs a figure. These are **not** applied here — they become proposal rows for
the propose → confirm → apply pass, which is what keeps the corpus human-annotated. A label a
script assigned and nobody confirmed would have to be disclosed as machine-assigned and excluded
from the kappa analysis.

In [ ]:
work = []

def _add(question, pid, field, old, why):
    work.append(dict(Patent_ID=pid, question=question, Field=field, old_value=old,
                     proposed_value="", confidence="", evidence=why, figures="", confirmed=""))

# ── A-9 — L5 sole-thrust: a TW with thrust on anything that does not tilt with the wing ──
NON_TILT = ["fuselage", "emp", "hull_array", "core_layout"]
for pid in df_conf.Patent_ID.unique():
    sub = df_conf[df_conf.Patent_ID == pid]
    if not (sub.Field == "topType").any():
        continue
    if strip_label(sub[sub.Field == "topType"].Value.iloc[0]) != "TW":
        continue
    hits = []
    for st in NON_TILT:
        c = sub[sub.Field == f"{st}_count"]
        n = pd.to_numeric(c.Value, errors="coerce").fillna(0).sum() if len(c) else 0
        if n > 0:
            hits.append(f"{st}={n:.0f}")
    for i in (1, 2, 3, 4):
        tl, cn = sub[sub.Field == f"wing{i}_tilt"], sub[sub.Field == f"wing{i}_count"]
        n = pd.to_numeric(cn.Value, errors="coerce").fillna(0).sum() if len(cn) else 0
        if len(tl) and strip_label(tl.Value.iloc[0]) == "Fixed" and n > 0:
            hits.append(f"wing{i}(non-tilting)={n:.0f}")
    bc = sub[sub.Field == "boom_count"]
    booms = pd.to_numeric(bc.Value, errors="coerce").fillna(0).sum() if len(bc) else 0
    if hits:
        _add("A-9 L5", pid, "topType", "TW",
             f"thrust on non-tilting structure: {', '.join(hits)}. Under the tightened L5 a "
             f"tilt-wing requires EVERY propulsor to tilt with the wing, so this is CVT. "
             f"NOTE: changing topType releases propKinLock, so propKin must be re-answered "
             f"per station — fix in the wizard, not the spreadsheet.")
    elif booms > 0:
        attach = sub[sub.Field.str.match(r"boom\d+_attach", na=False)].Value.map(strip_label).tolist()
        _add("A-9 L5 boom", pid, "topType", "TW",
             f"{booms:.0f} boom-mounted propulsors, boom attach={attach or 'n/a'}. "
             f"Does the boom tilt with the wing? Attached to the wing suggests yes (stays TW); "
             f"fixed to the fuselage means CVT.")

# ── A-13 — empennage vs fuselage under the new definition ───────────────────
for pid in df_conf.Patent_ID.unique():
    sub = df_conf[df_conf.Patent_ID == pid]
    c = sub[sub.Field == "emp_count"]
    n = pd.to_numeric(c.Value, errors="coerce").fillna(0).sum() if len(c) else 0
    if n <= 0:
        continue
    et = sub[sub.Field == "empType"]
    et_v = strip_label(et.Value.iloc[0]) if len(et) else "(none)"
    _add("A-13 empennage", pid, "emp_* mount", f"empennage ({n:.0f} propulsors)",
         f"empType={et_v}. An empennage bears stabilising surfaces; a bare rear fuselage does "
         f"not, and a propulsor on it is fuselage-mounted. A named tail shape suggests the "
         f"mount is already right, but empType is not independent evidence — check the figure.")

# ── Body motion — every record whose fusKin cannot stand as it is ────────────
for pid, v in fus_by_pid.items():
    top = top_by_pid.get(pid)
    if any(q["patent_id"] == pid for q in quarantined):
        continue                      # Section 6
    if top == "TB":
        _add("body motion", pid, "fusKin", v,
             "TB: the airframe reorients, so Fixed is impossible. Does the CABIN go with it "
             "(Tilting Body) or hang on its own joint and stay level (Variable Incidence)?")
    elif v in ("VarInc", "Variable"):
        _add("body motion", pid, "fusKin", v,
             f"legacy {v!r} on a {top} architecture. 'Variable' merged two current categories "
             f"and 'VarInc' was broader than today's Variable Incidence, so neither maps by rule.")

# ── The small ones ───────────────────────────────────────────────────────────
for pid in df_conf[(df_conf.Field == "fusShape")
                   & (df_conf.Value.map(strip_label) == "PodBoom")].Patent_ID:
    _add("fusShape", pid, "fusShape", "PodBoom",
         "Pod-and-Boom retired. Pick the base cross-section of the pod; the boom is already "
         "recorded as a boom group.")
for pid in df_conf[(df_conf.Field == "bgSty")
                   & (df_conf.Value.map(strip_label) == "Grid/Pattern")].Patent_ID:
    _add("bgSty", pid, "bgSty", "Grid/Pattern", "Retired v15.2. Solid Fill, Shaded/Gradient, or Other + note.")
for pid in df_conf[(df_conf.Field.str.endswith("_rmech", na=False))
                   & (df_conf.Value.map(strip_label) == "Retractable")].Patent_ID:
    _add("rmech", pid, "*_rmech", "Retractable",
         "Retractable split into Blade-folding (blades fold, unit stays) and Retracting "
         "(unit withdraws into the structure). Genuinely ambiguous — needs the figure.")
for pid in df_conf[(df_conf.Field.str.endswith("_propKin", na=False))
                   & (df_conf.Value.map(strip_label).isin(["Cyclic", "Vectored"]))].Patent_ID:
    _add("propKin", pid, "*_propKin", "Cyclic/Vectored",
         "Both retired v15.3. Fixed (unit does not pivot on its mount) or Tilt (it does).")

df_work = pd.DataFrame(work)
work_path = PROPOSALS_DIR / f"worklist_{BATCH}.csv"
df_work.to_csv(work_path, index=False)

print(f"{work_path.name}: {len(df_work)} decisions")
if len(df_work):
    print(df_work.question.value_counts().to_string())

## Section 9 — Write the conformed export

Backs up the current working copy first (`PRE_CONFORM_<timestamp>`), writes the conformed rows in
its place, and appends every applied change to `CORRECTIONS_LOG.csv`.

Only the deterministic changes land here. Worklist rows are untouched until they have been
confirmed — that pass writes them back through the same log.

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = src_xlsx.with_name(f"{src_xlsx.stem}.PRE_CONFORM_{stamp}.xlsx")
shutil.copy2(src_xlsx, backup)

df_out = df_conf[COLS].copy()
df_out.to_excel(src_xlsx, sheet_name="Review", index=False)

# CORRECTIONS_LOG.csv is the audit trail the thesis points at: one row per field per patent,
# with what decided it and why. Appended, never rewritten.
corr_path = CORRECTED_DIR / "CORRECTIONS_LOG.csv"
now = datetime.now(timezone.utc).isoformat(timespec="seconds")
corr = pd.DataFrame([
    dict(applied_utc=now, batch=BATCH, patent_id=h["patent_id"], section=h["section"],
         field=h["field"], old_value=h["old_value"], new_value=h["new_value"],
         rule=h["rule"], evidence=h["evidence"], confirmed_by="02a_legacy (deterministic)")
    for h in hand_rules
] + [
    dict(applied_utc=now, batch=BATCH, patent_id=f"({len(results)} patents)", section="G1/M1/M2/M3",
         field="(morphology)", old_value=f"{len(df_raw):,} rows as exported",
         new_value=f"{len(df_out):,} rows after v15_3 round-trip",
         rule="wizard round-trip (v15_3 ingest/export)",
         evidence=f"see harmonisation_log_{BATCH}.csv for the per-field diff",
         confirmed_by="02a_legacy (deterministic)")
])
corr.to_csv(corr_path, mode="a", header=not corr_path.exists() or corr_path.stat().st_size == 0,
            index=False)

print(f"backup   : {backup.name}")
print(f"written  : {src_xlsx.name}  ({len(df_out):,} rows)")
print(f"log      : {corr_path.name}  (+{len(corr)} rows)")
print(f"\nnext: run 02a_preprocessing.ipynb with sheet_name = {BATCH!r}")